# 🤖 Agenten-Verstärkungslernen — Jupyter Notebook

**Grundlagen des Reinforcement Learning: Q-Learning, Policy Gradient & DQN**

Dieses Notebook implementiert die drei fundamentalen RL-Algorithmen von Grund auf:
- 🧠 **Q-Learning (tabular)** — GridWorld mit Hindernissen und Cliff Walking
- 📈 **Policy Gradient (REINFORCE)** — 2-Layer-Netzwerk von Hand
- 🎮 **Deep Q-Network (DQN)** — CartPole mit Experience Replay

---

## 1. Setup & Importe

In [ ]:
import sys
import os
import numpy as np
import random
from collections import defaultdict, deque
from typing import List, Tuple, Optional
import matplotlib.pyplot as plt
from datetime import datetime

# Repository-Pfad hinzufügen
sys.path.insert(0, '/opt/data/agenten-verstaerkungslernen')

# Versuche, das echte rl_agent-Modul zu importieren
try:
    from rl_agent import GridWorld, QLearning, PolicyGradient
    print('✅ rl_agent-Modul importiert')
except ImportError as e:
    print(f'⚠️ rl_agent nicht importierbar: {e}')
    print('Verwende eigenständige Implementierung im Notebook.')

# Prüfe PyTorch-Verfügbarkeit
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_AVAILABLE = True
    print('✅ PyTorch verfügbar')
except ImportError:
    TORCH_AVAILABLE = False
    print('⚠️ PyTorch nicht installiert — DQN-Teil verwendet NumPy-Fallback')

# W&B
try:
    import wandb
    WANDB_AVAILABLE = True
    print('✅ W&B verfügbar')
except ImportError:
    WANDB_AVAILABLE = False
    print('⚠️ W&B nicht installiert')

print(f'📅 Ausführungszeit: {datetime.now().strftime("%d.%m.%Y %H:%M")}')
print(f'🐍 Python: {sys.version}')

---
## Teil 1: Q-Learning auf GridWorld

Tabular Q-Learning mit ε-greedy Exploration. Der Agent lernt, den kürzesten Weg zum Ziel zu finden.

### 1.1 GridWorld-Umgebung

In [ ]:
class GridWorld:
    """
    N×N Grid. Agent startet bei (0,0), Ziel bei (N-1,N-1).
    Aktionen: 0=hoch, 1=rechts, 2=runter, 3=links
    Belohnung: +1 am Ziel, -0.01 pro Schritt (Effizienz-Anreiz)
    """
    
    def __init__(self, size: int = 4, obstacles: Optional[List[tuple]] = None,
                 cliff: bool = False):
        self.size = size
        self.goal = (size - 1, size - 1)
        self.start = (0, 0)
        self.obstacles = set(obstacles or [])
        self.cliff = cliff
        self.reset()
    
    def reset(self):
        self.pos = self.start
        return self.pos
    
    def step(self, action: int) -> Tuple[tuple, float, bool]:
        r, c = self.pos
        if action == 0:    r = max(0, r - 1)
        elif action == 1:  c = min(self.size - 1, c + 1)
        elif action == 2:  r = min(self.size - 1, r + 1)
        elif action == 3:  c = max(0, c - 1)
        
        new_pos = (r, c)
        
        # Cliff: Absturz → Reset mit Strafe
        if self.cliff and r == self.size - 1 and 0 < c < self.size - 1:
            self.pos = self.start
            return self.pos, -1.0, False
        
        # Hindernis: Bleiben mit Strafe
        if new_pos in self.obstacles:
            return self.pos, -0.1, False
        
        self.pos = new_pos
        done = self.pos == self.goal
        reward = 1.0 if done else -0.01
        return self.pos, reward, done

# Test: GridWorld 4x4
env = GridWorld(size=4)
print(f'GridWorld {env.size}×{env.size} — Start: {env.start}, Ziel: {env.goal}')
print(f'Aktionen: 0=↑, 1=→, 2=↓, 3=←')

### 1.2 Q-Learning-Agent

In [ ]:
class QLearning:
    """Tabular Q-Learning mit ε-greedy Exploration."""
    
    def __init__(self, env: GridWorld, lr: float = 0.1,
                 gamma: float = 0.99, epsilon: float = 0.1):
        self.env = env
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon
        self.Q = defaultdict(lambda: np.zeros(4))
    
    def choose_action(self, state: tuple) -> int:
        if random.random() < self.epsilon:
            return random.randint(0, 3)
        return np.argmax(self.Q[state])
    
    def train(self, episodes: int = 1000) -> List[float]:
        """Trainiert den Agenten. Gibt Rewards pro Episode zurück."""
        rewards_history = []
        
        for ep in range(episodes):
            state = self.env.reset()
            total_reward = 0
            done = False
            
            while not done:
                action = self.choose_action(state)
                next_state, reward, done = self.env.step(action)
                
                # Q-Learning Update
                best_next = np.max(self.Q[next_state])
                td_target = reward + self.gamma * best_next * (1 - done)
                td_error = td_target - self.Q[state][action]
                self.Q[state][action] += self.lr * td_error
                
                state = next_state
                total_reward += reward
            
            rewards_history.append(total_reward)
            self.epsilon = max(0.01, self.epsilon * 0.995)
        
        return rewards_history
    
    def get_policy(self) -> np.ndarray:
        """Gibt die gelernte Policy als Grid zurück."""
        grid = np.zeros((self.env.size, self.env.size), dtype=int)
        for r in range(self.env.size):
            for c in range(self.env.size):
                if (r, c) == self.env.goal:
                    grid[r, c] = -1  # Goal
                else:
                    grid[r, c] = np.argmax(self.Q[(r, c)])
        return grid

print('✅ QLearning-Agent definiert')

### 1.3 Training & Visualisierung

In [ ]:
# ── Training auf Standard-GridWorld ──────────────────────
env_std = GridWorld(size=5)
agent_std = QLearning(env_std, lr=0.1, gamma=0.99, epsilon=0.3)
rewards_std = agent_std.train(episodes=500)

# ── Training mit Hindernissen ────────────────────────────
env_obs = GridWorld(size=5, obstacles=[(1,1), (2,2), (1,3), (3,1)])
agent_obs = QLearning(env_obs, lr=0.1, gamma=0.99, epsilon=0.3)
rewards_obs = agent_obs.train(episodes=500)

# ── Training auf Cliff ───────────────────────────────────
env_cliff = GridWorld(size=5, cliff=True)
agent_cliff = QLearning(env_cliff, lr=0.1, gamma=0.99, epsilon=0.3)
rewards_cliff = agent_cliff.train(episodes=500)

print('✅ Alle drei Umgebungen trainiert!')

In [ ]:
# ── Visualisierung ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
action_names = {0: '↑', 1: '→', 2: '↓', 3: '←', -1: '🎯'}

envs = [
    ('Standard', env_std, agent_std, rewards_std),
    ('Hindernisse', env_obs, agent_obs, rewards_obs),
    ('Cliff', env_cliff, agent_cliff, rewards_cliff),
]

for i, (name, env, agent, rewards) in enumerate(envs):
    # Policy-Grid
    ax = axes[0, i]
    policy = agent.get_policy()
    size = env.size
    
    for r in range(size):
        for c in range(size):
            if (r, c) == env.goal:
                bg, text = '#c8e6c9', '🎯'
            elif (r, c) in env.obstacles:
                bg, text = '#ffcdd2', '🧱'
            elif env.cliff and r == size-1 and 0 < c < size-1:
                bg, text = '#ffcdd2', '⚠️'
            else:
                bg, text = '#e3f2fd', action_names.get(int(policy[r, c]), '?')
            ax.text(c, size-1-r, text, ha='center', va='center', fontsize=18,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=bg, edgecolor='gray'))
    
    ax.set_xlim(-0.5, size-0.5)
    ax.set_ylim(-0.5, size-0.5)
    ax.set_xticks(range(size))
    ax.set_yticks(range(size))
    ax.set_yticklabels(range(size-1, -1, -1))
    ax.set_title(f'{name} — Gelernte Policy', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Reward-Verlauf
    ax = axes[1, i]
    ax.plot(rewards, color='#2196F3', alpha=0.7, linewidth=0.8)
    # Gleitender Durchschnitt
    window = 50
    if len(rewards) >= window:
        ma = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), ma, color='#e74c3c', linewidth=2, label=f'Ø {window}')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward')
    ax.set_title(f'{name} — Reward-Verlauf', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Teil 2: Policy Gradient (REINFORCE)

REINFORCE-Algorithmus mit einem manuellen 2-Layer-Netzwerk. Der Agent lernt eine Policy direkt durch Gradientenabstieg.

### 2.1 Policy-Gradient-Agent

In [ ]:
class PolicyGradient:
    """
    REINFORCE-Algorithmus mit einem einfachen 2-Layer-Netzwerk.
    Funktioniert für diskrete Action-Spaces.
    """
    
    def __init__(self, state_dim: int, action_dim: int,
                 lr: float = 0.01, gamma: float = 0.99):
        self.lr = lr
        self.gamma = gamma
        
        # Einfaches 2-Layer-Netzwerk
        self.W1 = np.random.randn(state_dim, 32) * 0.1
        self.b1 = np.zeros(32)
        self.W2 = np.random.randn(32, action_dim) * 0.1
        self.b2 = np.zeros(action_dim)
    
    def forward(self, state: np.ndarray) -> np.ndarray:
        """Forward-Pass: state → action probabilities."""
        h = np.maximum(0, state @ self.W1 + self.b1)  # ReLU
        logits = h @ self.W2 + self.b2
        # Softmax
        shifted = logits - np.max(logits)
        exp = np.exp(shifted)
        return exp / np.sum(exp)
    
    def sample_action(self, state: np.ndarray) -> Tuple[int, np.ndarray]:
        """Sample eine Aktion aus der Policy."""
        probs = self.forward(state)
        action = np.random.choice(len(probs), p=probs)
        return action, probs
    
    def update(self, episode: List[Tuple[np.ndarray, int, float]]):
        """REINFORCE Update."""
        # Discounted Returns berechnen
        returns = []
        G = 0
        for _, _, r in reversed(episode):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = np.array(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        for (state, action, _), G in zip(episode, returns):
            probs = self.forward(state)
            
            # Gradient für Cross-Entropy
            dlogits = probs.copy()
            dlogits[action] -= 1
            dlogits *= G
            
            # Backward (manuell)
            h = np.maximum(0, state @ self.W1 + self.b1)
            dh = dlogits @ self.W2.T
            dh[h <= 0] = 0
            
            # Update
            self.W2 -= self.lr * np.outer(h, dlogits)
            self.b2 -= self.lr * dlogits
            self.W1 -= self.lr * np.outer(state, dh)
            self.b1 -= self.lr * dh

print('✅ PolicyGradient-Agent definiert')

### 2.2 Training & Visualisierung

In [ ]:
# ── Training ─────────────────────────────────────────────
STATE_DIM = 4
ACTION_DIM = 2
EPISODES_PG = 300

pg = PolicyGradient(state_dim=STATE_DIM, action_dim=ACTION_DIM, lr=0.01, gamma=0.99)

rewards_pg = []
for ep in range(EPISODES_PG):
    episode_data = []
    state = np.random.randn(STATE_DIM) * 0.5
    
    for _ in range(20):
        action, probs = pg.sample_action(state)
        # Belohnung: Aktion 0 bevorzugt
        reward = 1.0 if action == 0 else -0.5
        reward += np.dot(state, np.ones(STATE_DIM)) * 0.1
        episode_data.append((state.copy(), action, reward))
        state = np.random.randn(STATE_DIM) * 0.5
    
    pg.update(episode_data)
    total_r = sum(r for _, _, r in episode_data)
    rewards_pg.append(total_r)

print(f'✅ Policy Gradient: {EPISODES_PG} Episoden trainiert')
print(f'   Finaler Reward: {rewards_pg[-1]:.3f}')
print(f'   Durchschnitt (letzte 50): {np.mean(rewards_pg[-50:]):.3f}')

In [ ]:
# ── Visualisierung ───────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Reward-Verlauf
ax1.plot(rewards_pg, color='#9b59b6', alpha=0.7, linewidth=0.8)
window = 30
ma = np.convolve(rewards_pg, np.ones(window)/window, mode='valid')
ax1.plot(range(window-1, len(rewards_pg)), ma, color='#e74c3c', linewidth=2, label=f'Ø {window}')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.set_title('Policy Gradient — Reward-Verlauf', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Finale Policy-Verteilung
test_states = np.random.randn(8, STATE_DIM) * 0.5
probs_0 = []
probs_1 = []
for s in test_states:
    p = pg.forward(s)
    probs_0.append(p[0])
    probs_1.append(p[1])

x = range(len(test_states))
ax2.bar([i - 0.15 for i in x], probs_0, 0.3, label='Aktion 0 (bevorzugt)', color='#2ecc71')
ax2.bar([i + 0.15 for i in x], probs_1, 0.3, label='Aktion 1', color='#e74c3c')
ax2.set_xlabel('Test-State')
ax2.set_ylabel('Wahrscheinlichkeit')
ax2.set_title('Finale Policy-Verteilung', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## Teil 3: Deep Q-Network (DQN) — CartPole

Deep Q-Network mit Experience Replay. CartPole-Simulation ohne externe Abhängigkeiten.

### 3.1 CartPole-Simulation

In [ ]:
class CartPoleSim:
    """Physik-basierte CartPole-Simulation (kein gymnasium nötig)."""
    
    def __init__(self):
        self.gravity = 9.8
        self.masscart = 1.0
        self.masspole = 0.1
        self.total_mass = self.masscart + self.masspole
        self.length = 0.5
        self.polemass_length = self.masspole * self.length
        self.tau = 0.02
        self.x_threshold = 2.4
        self.theta_threshold = 0.2095
        self.reset()
    
    def reset(self):
        self.x = np.random.uniform(-0.05, 0.05)
        self.x_dot = np.random.uniform(-0.05, 0.05)
        self.theta = np.random.uniform(-0.05, 0.05)
        self.theta_dot = np.random.uniform(-0.05, 0.05)
        return np.array([self.x, self.x_dot, self.theta, self.theta_dot])
    
    def step(self, action: int):
        force = 10.0 if action == 1 else -10.0
        
        temp = (force + self.polemass_length * self.theta_dot**2 * np.sin(self.theta)) / self.total_mass
        theta_acc = (self.gravity * np.sin(self.theta) - np.cos(self.theta) * temp) / (
            self.length * (4.0/3.0 - self.masspole * np.cos(self.theta)**2 / self.total_mass)
        )
        x_acc = temp - self.polemass_length * theta_acc * np.cos(self.theta) / self.total_mass
        
        self.x += self.tau * self.x_dot
        self.x_dot += self.tau * x_acc
        self.theta += self.tau * self.theta_dot
        self.theta_dot += self.tau * theta_acc
        
        done = abs(self.x) > self.x_threshold or abs(self.theta) > self.theta_threshold
        reward = 1.0 if not done else 0.0
        return np.array([self.x, self.x_dot, self.theta, self.theta_dot]), reward, done

print('✅ CartPoleSim definiert')

### 3.2 DQN-Agent (NumPy-basiert)

In [ ]:
class SimpleDQN:
    """Einfaches 3-Layer-Netzwerk für Q-Wert-Approximation (NumPy)."""
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 64, lr: float = 0.001):
        self.W1 = np.random.randn(state_dim, hidden_dim) * 0.1
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, hidden_dim) * 0.1
        self.b2 = np.zeros(hidden_dim)
        self.W3 = np.random.randn(hidden_dim, action_dim) * 0.1
        self.b3 = np.zeros(action_dim)
        self.lr = lr
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        self.z1 = x @ self.W1 + self.b1
        self.a1 = np.maximum(0, self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = np.maximum(0, self.z2)
        self.z3 = self.a2 @ self.W3 + self.b3
        return self.z3
    
    def update(self, x: np.ndarray, y: np.ndarray):
        pred = self.forward(x)
        error = pred - y
        
        dW3 = np.outer(self.a2, error)
        db3 = error
        da2 = error @ self.W3.T
        dz2 = da2 * (self.z2 > 0)
        dW2 = np.outer(self.a1, dz2)
        db2 = dz2
        da1 = dz2 @ self.W2.T
        dz1 = da1 * (self.z1 > 0)
        dW1 = np.outer(x, dz1)
        db1 = dz1
        
        self.W3 -= self.lr * dW3
        self.b3 -= self.lr * db3
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

print('✅ SimpleDQN-Agent definiert')

### 3.3 DQN-Training

In [ ]:
# ── Hyperparameter ────────────────────────────────────────
DQN_EPISODES = 300
GAMMA = 0.99
EPSILON_START = 1.0
EPSILON_DECAY = 0.995
EPSILON_MIN = 0.01
BATCH_SIZE = 32
MEMORY_SIZE = 2000
HIDDEN_DIM = 64
LR = 0.001

# ── Initialisierung ───────────────────────────────────────
env = CartPoleSim()
agent = SimpleDQN(state_dim=4, action_dim=2, hidden_dim=HIDDEN_DIM, lr=LR)
memory = []
epsilon = EPSILON_START
rewards_dqn = []

print(f'🚀 DQN-Training: {DQN_EPISODES} Episoden...')

for ep in range(DQN_EPISODES):
    state = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        # Epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, 1)
        else:
            q_vals = agent.forward(state)
            action = int(np.argmax(q_vals))
        
        next_state, reward, done = env.step(action)
        memory.append((state, action, reward, next_state, done))
        if len(memory) > MEMORY_SIZE:
            memory.pop(0)
        
        # Experience Replay
        if len(memory) >= BATCH_SIZE:
            batch = random.sample(memory, BATCH_SIZE)
            for s, a, r, ns, d in batch:
                target = r
                if not d:
                    target += GAMMA * np.max(agent.forward(ns))
                q_vals = agent.forward(s)
                q_vals[a] = target
                agent.update(s, q_vals)
        
        state = next_state
        total_reward += reward
    
    rewards_dqn.append(total_reward)
    epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
    
    if ep % 50 == 0:
        avg = np.mean(rewards_dqn[-50:]) if len(rewards_dqn) >= 50 else np.mean(rewards_dqn)
        print(f'  Episode {ep:3d}/{DQN_EPISODES} — Reward: {total_reward:6.1f} — Ø50: {avg:6.1f} — ε: {epsilon:.3f}')

print(f'\n✅ DQN-Training abgeschlossen!')
print(f'   Beste Episode: {max(rewards_dqn):.0f}')
print(f'   Durchschnitt (letzte 50): {np.mean(rewards_dqn[-50:]):.1f}')

### 3.4 DQN-Visualisierung

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Reward-Verlauf
ax1.plot(rewards_dqn, color='#2196F3', alpha=0.5, linewidth=0.6, label='Episode')
window = 30
ma = np.convolve(rewards_dqn, np.ones(window)/window, mode='valid')
ax1.plot(range(window-1, len(rewards_dqn)), ma, color='#e74c3c', linewidth=2, label=f'Ø {window}')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Reward (Schritte)')
ax1.set_title('DQN CartPole — Reward-Verlauf', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Epsilon-Decay
epsilons = [max(EPSILON_MIN, EPSILON_START * (EPSILON_DECAY ** i)) for i in range(DQN_EPISODES)]
ax2.plot(epsilons, color='#9b59b6', linewidth=2)
ax2.set_xlabel('Episode')
ax2.set_ylabel('Epsilon')
ax2.set_title('Exploration-Rate (ε-Decay)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0.1, color='red', linestyle='--', alpha=0.5, label='ε = 0.1')
ax2.legend()

plt.tight_layout()
plt.show()

---
## Teil 4: Algorithmen-Vergleich

Vergleich aller drei RL-Algorithmen hinsichtlich Lernkurve und Performance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets = [
    (rewards_std, 'Q-Learning (GridWorld)', '#2196F3'),
    (rewards_pg, 'Policy Gradient (REINFORCE)', '#9b59b6'),
    (rewards_dqn, 'DQN (CartPole)', '#2ecc71'),
]

for i, (data, title, color) in enumerate(datasets):
    ax = axes[i]
    ax.plot(data, color=color, alpha=0.5, linewidth=0.6)
    window = max(5, len(data) // 10)
    ma = np.convolve(data, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(data)), ma, color='#e74c3c', linewidth=2, label=f'Ø {window}')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Finale Metriken
    final_avg = np.mean(data[-50:])
    ax.text(0.95, 0.05, f'Ø letzte 50: {final_avg:.2f}',
            transform=ax.transAxes, ha='right', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('RL-Algorithmen im Vergleich', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Zusammenfassung

| Algorithmus | Umgebung | State-Space | Besonderheit |
|---|---|---|---|
| **Q-Learning** | GridWorld (5×5) | Diskret (25 States) | Tabular, ε-greedy |
| **Policy Gradient** | Synthetisch (4D) | Kontinuierlich | REINFORCE, manuelles Netzwerk |
| **DQN** | CartPole | Kontinuierlich (4D) | Experience Replay, ε-Decay |

### Kernkonzepte

- **Q-Learning**: Lernt Q-Werte für State-Action-Paare. Update: `Q(s,a) += α[r + γ·max Q(s') - Q(s,a)]`
- **Policy Gradient**: Optimiert Policy direkt. Update via Gradienten der erwarteten Belohnung.
- **DQN**: Approximiert Q-Werte mit neuronalem Netz. Nutzt Experience Replay für stabiles Training.

---

**Nächste Schritte:**
- Echte PyTorch-DQN mit Double DQN & Target Network (`rl_agent.py`)
- W&B-Tracking aktivieren: `export WANDB_API_KEY=...`
- Hyperparameter-Sweeps: `python sweep_runner.py`
- Streamlit-App starten: `streamlit run app/app.py`
- CLI-Training: `python train_rl.py --algo dqn --episodes 1000`